# Milestone 1

This milestone has been created to familiarize you with the process of Exploratory Data Analysis (EDA), text processing, and establishing baseline similarity metrics required specifically for Natural Language Processing (NLP) pipelines.

Suggested Readings:
 - Tokenization & Text Normalization
 - Stop Words & Vocabulary Constraints
 - TF-IDF (Term Frequency-Inverse Document Frequency)
 - Cosine Similarity in NLP
 - Understanding Mean Average Precision (MAP@3)

Perform the following tasks on the dataset provided as a part of the Kaggle competition.
Competition link: https://www.kaggle.com/competitions/smart-mcq-solver-challenge

In [98]:
import pandas as pd
import numpy as np


# Loading the datasets
df = pd.read_csv("../../data/raw/train.csv")
test_df = pd.read_csv("../../data/raw/test.csv")

Calculate the frequency distribution of the correct  answer  (A, B, C, D, E) in train.csv. Based on your counts, what is the sum of the occurrences of the most frequent option and the least frequent option?  

In [99]:
print(df['answer'].value_counts().sort_index())

print(df['answer'].value_counts().max()+df['answer'].value_counts().min())

answer
A    369
B    490
C    459
D    358
E    324
Name: count, dtype: int64
814


After converting the prompt column to lowercase and removing all standard punctuation characters (using Python's string.punctuation), split the text by whitespace. What is the total number of unique words (vocabulary size) across the entire cleaned prompt column of train.csv?  

In [100]:
def to_lower_and_remove_punctuation(text):
    import string
    text=str(text).lower()
    text=text.translate(str.maketrans('', '', string.punctuation))
    return text

cleaned_prompt = df['prompt'].apply(to_lower_and_remove_punctuation)

unique_words = set()
for text in cleaned_prompt:
    unique_words.update(text.split())

print(f"Total unique words: {len(unique_words)}")

Total unique words: 859


Using the cleaned prompt from Row ID 1, filter out the standard English stop words using sklearn.feature_extraction.text.ENGLISH_STOP_WORDS. How many words are left in the prompt for Row ID 1 after filtering?

In [101]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

words = cleaned_prompt.iloc[0].split()

filtered_words=[]

for word in words:
    if word not in ENGLISH_STOP_WORDS:
        filtered_words.append(word)

print(filtered_words)
len(filtered_words)

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']


13

Fit a default TfidfVectorizer(stop_words='english') on a list containing all the combined text of the prompts and options in train.csv. What is the exact total number of feature columns (vocabulary size) generated by the vectorizer?

In [102]:
from sklearn.feature_extraction.text import TfidfVectorizer

combined_text=df['prompt']+ " " + df['A'] + " " + df['B'] + " " + df['C'] + " " + df['D'] + " " + df['E']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_mat=tfidf.fit_transform(combined_text)
tfidf_mat.shape[1]


2762

Using the TF-IDF vectorizer fitted in Question 3, calculate the cosine similarity between the prompt and option A strictly for Row ID 1. What is the resulting similarity score? (Round to 4 decimal places).

In [110]:
row1=df.iloc[0]
prompt_v=tfidf.transform([row1['prompt']]) # type: ignore
option_a_v=tfidf.transform([row1['A']])

from sklearn.metrics.pairwise import cosine_similarity
round(cosine_similarity(prompt_v, option_a_v)[0][0], 4)

np.float64(0.272)

Expand the logic from Question 4: For every row in train.csv, calculate the cosine similarity between the prompt and each of its 5 options .  Then calculate the percentage of instances where the option with the highest cosine similarity matches the correct answer.

In [104]:
prompt_vectors=tfidf.transform(df['prompt'])
option_a_vectors=tfidf.transform(df['A'])
option_b_vectors=tfidf.transform(df['B'])
option_c_vectors=tfidf.transform(df['C'])
option_d_vectors=tfidf.transform(df['D'])
option_e_vectors=tfidf.transform(df['E'])
similarity=pd.DataFrame()

similarity['A']=cosine_similarity(prompt_vectors, option_a_vectors).diagonal()
similarity['B']=cosine_similarity(prompt_vectors, option_b_vectors).diagonal()
similarity['C']=cosine_similarity(prompt_vectors, option_c_vectors).diagonal()
similarity['D']=cosine_similarity(prompt_vectors, option_d_vectors).diagonal()
similarity['E']=cosine_similarity(prompt_vectors, option_e_vectors).diagonal()

highest_similarity_option=similarity.idxmax(axis=1)
accuracy=(highest_similarity_option==df['answer']).mean()
print(f"{accuracy*100:.4f}%")

13.5500%


If the ground truth answer for a question is C, what is the MAP@3 score if a model predicts C A B?

In [105]:
def average_prediction_score3(prediction:list, ground_truth:str) -> float:
    if not prediction:
        return 0.0
    score = 0.0
    for i, pred in enumerate(prediction[:3]):
        if pred == ground_truth:
            score += 1 / (i + 1)
    return score

average_prediction_score3(['C','A','B'], 'C')

1.0

If the ground truth answer for a question is  B, what is the MAP@3 score if a model predicts D B E?  


In [106]:
average_prediction_score3(['D','B','E'], 'B')

0.5

The Majority Class Baseline: 

Find the most frequent correct answer in the training set (using your data from Q1). Make a static prediction for every single row where that most frequent answer is your 1st guess, followed by the second most frequent, and then the third most frequent. What is the overall MAP@3 score of this "Majority Class" baseline on train.csv?

In [107]:
static_ans=df['answer'].value_counts().index.tolist()[:3]

score=0.0
for ans in df['answer']:
    score+=average_prediction_score3(static_ans, ans)
score/=len(df)
print(f"Average Prediction Score: {score:.4f}")

Average Prediction Score: 0.4213


The TF-IDF Pipeline: 

Build a basic pipeline that evaluates every row in train.csv. For each row, calculate the TF-IDF cosine similarity between the prompt and each of the 5 options. Sort these options from highest similarity to lowest to form your top 3 predictions. What is the final average MAP@3 score of this TF-IDF pipeline across the entire training set?

In [108]:
top3_preds = similarity.apply(
    lambda r: r.sort_values(ascending=False).index[:3].tolist(),
    axis=1
)
total_score=0.0
for pred, true in zip(top3_preds, df['answer']):
    total_score+=average_prediction_score3(pred, true)
print(f"Average Prediction Score: {total_score/len(df):.4f}")

Average Prediction Score: 0.2947
